# 🎯 [WEL03_M04] 복합 오염원 유입 징후 및 급수 안전도 위험 등급 분류 머신러닝 모델 고도화
### Google Colab 실습 환경 (장병 복지·지원 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/WEL03/wel03_m04_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 장병 복지·지원 (Level 2) |
| **미션 ID** | WEL03_M04 |
| **미션 주제** | 복합 오염원 유입 징후 및 급수 안전도 위험 등급 분류 머신러닝 모델 고도화 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
전장에서는 급수차, 정수 장비, 우물, 하천 취수원 등 급수원의 수질 이상을 빨리 알아채지 못하면 장병의 건강과 부대 활동에 바로 영향을 주지만, 센서 값은 잡음과 고장이 많아 오경보로 불필요하게 급수를 끊는 일도 생긴다. 러시아-우크라이나 전쟁 사례에서 학습 초점으로 삼은 전장 위생·건강 위험 관리를 반영해, 본 프로젝트는 가상 수질검사 데이터를 분석해 위험 신호를 데이터로 판단하는 과제를 다룬다. 교육용 가상 급수원 8곳의 pH, 탁도, 잔류염소, 중금속, 대장균 측정 자료를 코드로 만들어 스키마 정의, 센서 결측·이상치 정제, 기준치 Baseline 판정, 복합 오염 징후 분류 모델, 신뢰도 지수와 오경보 감축, 상황판 대시보드, 급수 중단 HITL 결재선, 대량 오염 부하 시험, AAR 임계치 재조정, 통합 패키지까지 단계별로 구현한다. 최종 목표는 수질 위험 신호를 근거와 함께 빠르게 제시하되, 실제 음용 판정과 급수 중단은 공식 기준과 의무(예방의학) 담당 확인, 지휘관 결심에 따르도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/wel03_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/ui/`, `src/hitl/` | 전처리, 모델, 대시보드, HITL 결재 등 핵심 파이썬 소스 코드 모듈 |
| **`docs/`** | `docs/wel03_data_spec.md`, `docs/wel03_aar_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서 등 실무/지휘관 문서 |
| **`configs/`**| `configs/wel03_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/wel03_stress_test.py` | 500% 부하 시험 및 pytest 단위/회귀 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (추천):** 코드 셀 첫 줄에 `%%writefile src/models/wel03_baseline_model.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 04. 복합 오염원 유입 징후 및 급수 안전도 위험 등급 분류 머신러닝 모델 고도화
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

복합 오염원 유입 징후 및 급수 위험 등급 분류 머신러닝 모델을 고도화한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **복합 오염원 유입 징후와 급수 안전도 위험 등급을 분류하는 머신러닝 모델을 고도화하고 방어 로직을 검증한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **변화량, 24시간 기준선 대비 편차, 6시간 변동성, 대장균 누적, 센서 처리 표시 등 29개 특징으로 랜덤 포레스트를 학습하고 Baseline과 비교 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/models/wel03_advanced_detector.py 및 예외 탐지 테스트 결과` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 기만/교란 공격에 대한 방어 로직의 현실성 및 신뢰성

#### 미션 PASS 조건

- 정량 기준: 복합 위협 탐지 재현율(Recall) 85% 이상, 처리 지연 200ms 이하


- 진행 기준: 복합 위협 시나리오 10종에 대한 탐지 테스트 통과


- 검증 방법: 합성 위협 데이터셋 주입 시 이상치 탐지 정확도 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""시계열 특징(변화량·기준선 편차·변동성·대장균 누적·센서 처리 표시)으로 급수 안전도 위험 등급(안전·주의·위험)을 분류하는 랜덤 포레스트 모델을 학습하고, 복합 위협 시나리오 10종(오염 사건 7종, 센서 교란 3종)으로 재현율·오경보·처리 지연을 검증한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

# ---------------------------------------------------------------------------
# [교육용가상자료] 야전 급수원 가상 수질 센서 데이터 생성기 (실제 부대·위치·검사 결과 없음)
# 가상 급수원 8곳 x 14일 x 1시간 간격. 실제 수질(true_*)·센서 측정값·오염 사건·센서 고장 정답을 함께 만든다.
# 기준치는 교육용 가정값이며, 실제 음용 판정은 공인 검사 결과와 의무(예방의학) 담당 기준에 따른다.
# ---------------------------------------------------------------------------
PARAMS = ["ph", "turbidity", "chlorine", "metal", "ecoli"]
SOURCES = [("W01", "water_truck", 1.0, 2.0), ("W02", "water_truck", 3.5, 1.2), ("W03", "purification_unit", 2.2, 4.1),
           ("W04", "purification_unit", 5.0, 3.3), ("W05", "well", 0.8, 5.6), ("W06", "well", 4.4, 6.0),
           ("W07", "river_intake", 6.3, 1.8), ("W08", "river_intake", 6.9, 4.9)]
BASE = {"water_truck": (7.2, 0.25, 0.75, 1.5), "purification_unit": (7.0, 0.20, 0.85, 1.2),
        "well": (6.8, 0.45, 0.50, 3.0), "river_intake": (7.3, 0.55, 0.60, 2.0)}   # pH, 탁도(NTU), 잔류염소(mg/L), 중금속(ug/L)
LIMITS = {"ph": (5.8, 8.5), "turbidity": (1.0, 2.0), "chlorine": (0.2, 0.1), "metal": (5.0, 10.0), "ecoli": (1, 1)}  # (주의, 위험) 가정 기준
EVENT_TYPES = {  # 오염 사건: 변수별 최대 변화량, 최대치 도달 시간(h)
    "sewage": ({"turbidity": 2.8, "chlorine": -0.5, "ecoli": 30, "ph": -0.3}, 4),
    "metal": ({"metal": 22.0, "ph": -0.5}, 4),
    "disinfect_fail": ({"chlorine": -0.8, "ecoli": 6}, 6),
    "chemical": ({"ph": 1.7, "turbidity": 1.0}, 3),
    "creeping_metal": ({"metal": 13.0}, 18),
}
FAULT_TYPES = ["stuck", "spike", "drift", "dropout"]
NOISE = {"ph": 0.04, "turbidity": 0.06, "chlorine": 0.03, "metal": 0.35}
DIGITS = {"ph": 2, "turbidity": 2, "chlorine": 2, "metal": 1}


def make_dataset(days=14, seed=7, n_events=6, n_faults=10, events=None, faults=None, sources=None):
    """events: [(급수원 번호, 사건 유형, 시작 시각 h, 지속 h, 강도)], faults: [(급수원 번호, 고장 유형, 항목, 시작 h, 지속 h)]"""
    rng = np.random.default_rng(seed)
    sources = sources or SOURCES
    T = days * 24
    tt = np.arange(T)
    hour = tt % 24
    night = (hour < 6) | (hour >= 20)
    rain = np.zeros(T)
    for _ in range(max(1, days // 4)):
        s0, d = int(rng.integers(0, T - 14)), int(rng.integers(6, 14))
        rain[s0:s0 + d] = np.maximum(rain[s0:s0 + d], rng.uniform(0.6, 1.0))
    if events is None:
        events = [(int(rng.integers(len(sources))), str(rng.choice(list(EVENT_TYPES))), int(rng.integers(min(24, T // 4), max(T // 4 + 1, T - 48))),
                   int(rng.integers(10, 30)), 1.0) for _ in range(n_events)]
    if faults is None:
        faults = [(int(rng.integers(len(sources))), str(rng.choice(FAULT_TYPES)), str(rng.choice(["ph", "turbidity", "chlorine", "metal"])),
                   int(rng.integers(12, max(13, T - 30))), int(rng.integers(6, 24))) for _ in range(n_faults)]
    out = []
    for si, (sid, stype, x, y) in enumerate(sources):
        ph0, tb0, cl0, mt0 = BASE[stype]
        tr = {"ph": ph0 + 0.08 * np.sin(2 * np.pi * hour / 24) + rng.normal(0, 0.03, T),
              "turbidity": tb0 * (1 + 0.1 * rng.normal(0, 1, T)),
              "chlorine": (cl0 * np.exp(-0.022 * hour) if stype == "water_truck" else cl0 + rng.normal(0, 0.03, T)),
              "metal": mt0 + rng.normal(0, 0.2, T), "ecoli": np.zeros(T)}
        if stype in ("river_intake", "well"):
            tr["turbidity"] = tr["turbidity"] + rain * (0.9 if stype == "river_intake" else 0.45)
        f_all, etype = np.zeros(T), np.array(["none"] * T, dtype=object)
        for (s_i, et, onset, dur, scale) in events:
            if s_i != si:
                continue
            delta, ramp = EVENT_TYPES[et]
            f = np.clip((tt - onset + 1) / ramp, 0, 1) * ((tt >= onset) & (tt < onset + dur + ramp))
            for k, v in delta.items():
                tr[k] = tr[k] + v * f * scale
            etype[f > 0] = et
            f_all = np.maximum(f_all, f)
        tr["chlorine"] = np.clip(tr["chlorine"], 0.02, None)
        tr["turbidity"] = np.clip(tr["turbidity"], 0.05, None)
        tr["metal"] = np.clip(tr["metal"], 0.1, None)
        grade = np.where(f_all >= 0.2, 2, 0)
        mild = (tr["turbidity"] > LIMITS["turbidity"][0]) | (tr["chlorine"] < LIMITS["chlorine"][0]) | (tr["metal"] > LIMITS["metal"][0])
        grade = np.where((grade == 0) & mild, 1, grade)
        ms = {}
        for p in ["ph", "turbidity", "chlorine", "metal"]:
            sd = NOISE[p] * np.where(night, 1.5, 1.0) * (tr[p] if p == "turbidity" else 1)
            ms[p] = np.round(np.clip(tr[p] + rng.normal(0, 1, T) * sd, 0, None), DIGITS[p])
        ms["ecoli"] = (rng.poisson(tr["ecoli"]) + (rng.random(T) < 0.004)).astype(float)
        fault, fparam = np.array(["none"] * T, dtype=object), np.array(["-"] * T, dtype=object)
        for (s_i, ft, p, s0, d) in faults:
            if s_i != si:
                continue
            seg = slice(s0, min(T, s0 + d))
            if ft == "stuck":
                ms[p][seg] = ms[p][s0]
            elif ft == "spike":
                seg = slice(s0, s0 + 1)
                ms[p][s0] = {"ph": 11.8, "turbidity": 38.0, "chlorine": 0.0, "metal": 75.0}[p]
            elif ft == "drift":
                ramp = np.linspace(0, 1, seg.stop - s0)
                ms[p][seg] = np.clip(ms[p][seg] + ramp * {"ph": -1.6, "turbidity": 2.6, "chlorine": -0.75, "metal": 12.0}[p], 0, None).round(DIGITS[p])
            elif ft == "dropout":
                for q in PARAMS:
                    ms[q][seg] = np.nan
                p = "all"
            fault[seg], fparam[seg] = ft, p
        df = pd.DataFrame({"source_id": sid, "source_type": stype, "x_km": x, "y_km": y, "t": tt,
                           "cond": np.where(rain > 0, "storm", np.where(night, "night", "day")),
                           **{p: ms[p] for p in PARAMS}, **{f"true_{p}": tr[p] for p in PARAMS},
                           "event": etype, "fault": fault, "fault_param": fparam, "grade": grade})
        out.append(df)
    return pd.concat(out, ignore_index=True)
# ---------------------------------------------------------------------------

# ---- 전처리 공통 함수: 물리 범위 점검 -> 고정값(stuck) 표시 -> 단발 스파이크 제거 -> 결측 보정 ----
PHYS = {"ph": (0, 14), "turbidity": (0, 1000), "chlorine": (0, 10), "metal": (0, 1000), "ecoli": (0, 100000)}
SPIKE_K = {"day": 6.0, "night": 7.0, "storm": 9.0}          # 상황별(주간/야간/악천후) 스파이크 판정 배수
MIN_JUMP = {"ph": 0.8, "turbidity": 1.5, "chlorine": 0.3, "metal": 6.0}
STUCK_RUN, SHORT_GAP = 6, 3


def clean(df, spike_k=None, min_jump=None):
    """급수원별 시계열을 정제하고 항목별 처리 내역(<항목>_flag: range/stuck/spike/interp/long_gap)을 남긴다."""
    spike_k = spike_k or SPIKE_K
    min_jump = MIN_JUMP if min_jump is None else min_jump
    out = df.sort_values(["source_id", "t"]).reset_index(drop=True)
    for sid, idx in out.groupby("source_id").groups.items():
        idx = np.asarray(idx)
        kk = out.loc[idx, "cond"].map(spike_k).to_numpy(float)
        for p in PARAMS:
            v = out.loc[idx, p].to_numpy(float).copy()
            fl = np.array([""] * len(v), dtype=object)
            lo, hi = PHYS[p]
            bad = (v < lo) | (v > hi)
            v[bad], fl[bad] = np.nan, "range"
            if p != "ecoli":
                chg = np.r_[True, v[1:] != v[:-1]]                       # 같은 값이 STUCK_RUN시간 이상 반복 = 고정값 고장 의심
                rid = np.cumsum(chg)
                rlen = np.bincount(rid)[rid]
                fl[(rlen >= STUCK_RUN) & ~chg & ~np.isnan(v)] = "stuck"
                d = np.abs(np.diff(v[~np.isnan(v)]))
                sig = 1.4826 * np.median(d) + 1e-6 if len(d) else 1.0
                up, dn = v - np.r_[np.nan, v[:-1]], v - np.r_[v[1:], np.nan]
                thr = np.maximum(kk * sig, min_jump[p])
                spike = (np.abs(up) > thr) & (np.abs(dn) > thr) & (np.sign(up) == np.sign(dn))   # 튀었다가 바로 돌아온 값만
                v[spike], fl[spike] = np.nan, "spike"
            s = pd.Series(v)
            na = s.isna().to_numpy()
            grp = np.cumsum(np.r_[True, na[1:] != na[:-1]])
            glen = pd.Series(na.astype(int)).groupby(grp).transform("sum").to_numpy()
            lin, ff = s.interpolate(limit_area="inside").to_numpy(), s.ffill().bfill().to_numpy()
            short, longg = na & (glen <= SHORT_GAP), na & (glen > SHORT_GAP)
            v = np.where(short, np.where(np.isnan(lin), ff, lin), np.where(longg, ff, v))
            fl[short & (fl == "")] = "interp"
            fl[longg] = "long_gap"
            out.loc[idx, p] = v
            out.loc[idx, f"{p}_flag"] = fl
    return out

# ---- 가정 기준치 판정(Baseline)과 시계열 특징 공통 함수 ----
GRADE_NAME = {0: "안전", 1: "주의", 2: "위험"}


def rule_grade(df):
    """기준치 1차 필터: 위험(음용 부적합 의심) = 한 항목이라도 위험 기준 초과, 주의 = 주의 기준 초과."""
    ph, tb, cl, mt, ec = (df[p].to_numpy(float) for p in PARAMS)
    danger = (ph < LIMITS["ph"][0]) | (ph > LIMITS["ph"][1]) | (tb > LIMITS["turbidity"][1]) | (cl < LIMITS["chlorine"][1]) \
        | (mt > LIMITS["metal"][1]) | (ec >= LIMITS["ecoli"][1])
    caution = (tb > LIMITS["turbidity"][0]) | (cl < LIMITS["chlorine"][0]) | (mt > LIMITS["metal"][0])
    return np.where(danger, 2, np.where(caution, 1, 0))


def reasons(row):
    r = []
    if not LIMITS["ph"][0] <= row["ph"] <= LIMITS["ph"][1]:
        r.append(f"pH {row['ph']:.2f}")
    if row["turbidity"] > LIMITS["turbidity"][0]:
        r.append(f"탁도 {row['turbidity']:.2f}NTU")
    if row["chlorine"] < LIMITS["chlorine"][0]:
        r.append(f"잔류염소 {row['chlorine']:.2f}mg/L")
    if row["metal"] > LIMITS["metal"][0]:
        r.append(f"중금속 {row['metal']:.1f}ug/L")
    if row["ecoli"] >= 1:
        r.append(f"대장균 {row['ecoli']:.0f}")
    return ", ".join(r) or "-"


FEATS = [f"{p}{s}" for p in ["ph", "turbidity", "chlorine", "metal"] for s in ["", "_d1", "_d3", "_dev", "_sd6"]] + \
        ["ecoli", "ecoli_3h", "storm", "night", "n_flag", "truck", "purif", "well", "river"]


def add_features(c):
    """시간 순서를 지키는(과거 값만 쓰는) 급수원별 특징: 변화량, 24시간 기준선 대비 편차, 6시간 변동성, 대장균 누적 등."""
    c = c.sort_values(["source_id", "t"]).reset_index(drop=True)
    g = c.groupby("source_id")
    for p in ["ph", "turbidity", "chlorine", "metal"]:
        c[f"{p}_d1"] = g[p].diff().fillna(0)
        c[f"{p}_d3"] = g[p].diff(3).fillna(0)
        base = g[p].transform(lambda s: s.shift(1).rolling(24, min_periods=4).median())
        c[f"{p}_dev"] = (c[p] - base).fillna(0)
        c[f"{p}_sd6"] = g[p].transform(lambda s: s.rolling(6, min_periods=2).std()).fillna(0)
    c["ecoli_3h"] = g["ecoli"].transform(lambda s: s.rolling(3, min_periods=1).sum())
    c["storm"], c["night"] = (c["cond"] == "storm").astype(int), (c["cond"] == "night").astype(int)
    flags = [f"{p}_flag" for p in PARAMS if f"{p}_flag" in c]
    c["n_flag"] = sum(c[f].isin(["stuck", "spike", "long_gap", "range"]).astype(int) for f in flags) if flags else 0
    for k, name in [("truck", "water_truck"), ("purif", "purification_unit"), ("well", "well"), ("river", "river_intake")]:
        c[k] = (c["source_type"] == name).astype(int)
    return c


def prepare(df):
    c = add_features(clean(df))
    c["rule"] = rule_grade(c)
    return c


# 1) 학습(가상 2주 x 2세트)과 평가(벤치마크 seed 11)
train = pd.concat([prepare(make_dataset(seed=s, n_events=14, n_faults=14)) for s in (7, 8)], ignore_index=True)
test = prepare(make_dataset(seed=11, n_events=10, n_faults=14))
model = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced_subsample", random_state=0, n_jobs=1)
model.fit(train[FEATS], train["grade"])
t0 = time.perf_counter()
test["ml"] = model.predict(test[FEATS])
batch_ms = (time.perf_counter() - t0) * 1000
t0 = time.perf_counter()
for i in range(20):
    model.predict(test[FEATS].iloc[[i]])
single_ms = (time.perf_counter() - t0) / 20 * 1000

d = test["grade"] == 2
print(f"[교육용가상자료] 학습 {len(train):,}건 / 평가 {len(test):,}건, 특징 {len(FEATS)}개")
for name, col in [("Baseline 기준치", "rule"), ("랜덤 포레스트", "ml")]:
    p = test[col]
    print(f"{name}: 정확도 {(p == test['grade']).mean():.1%}, 위험 재현율 {((p == 2) & d).sum() / d.sum():.1%}, "
          f"위험 오경보 {int(((p == 2) & ~d).sum())}건")
rec = ((test["ml"] == 2) & d).sum() / d.sum()
print(f"복합 위협 재현율 {rec:.1%} (목표 85% 이상: {'달성' if rec >= 0.85 else '미달'}), "
      f"처리 지연 1건 {single_ms:.1f}ms / 전체 {batch_ms:.0f}ms (목표 200ms 이하: {'달성' if single_ms <= 200 else '미달'})")
print("Baseline이 놓친 서서히 오르는 중금속(creeping_metal) 중 모델 검출:",
      f"{int(((test['ml'] == 2) & d & (test['rule'] != 2) & (test['event'] == 'creeping_metal')).sum())}"
      f"/{int((d & (test['rule'] != 2) & (test['event'] == 'creeping_metal')).sum())}시간")

# 2) 오경보 유발 패턴: 위험 오경보가 어떤 센서 고장에서 나왔는가 -> 특징 공간 재설계 단서
fa = test[(test["ml"] == 2) & ~d]
print("모델 위험 오경보의 센서 고장 유형:", fa["fault"].value_counts().to_dict(), "-> drift(서서히 틀어짐)는 미션 5 교차 검증으로 보완")
imp = pd.Series(model.feature_importances_, FEATS).sort_values(ascending=False).head(6)
print("중요 특징 Top6:", ", ".join(f"{k} {v:.2f}" for k, v in imp.items()))

# 3) 복합 위협 시나리오 10종: (급수원, 사건) / 센서 교란은 오염이 없으므로 위험 경보가 나오면 안 된다
SCEN = [("하수 유입", [(6, "sewage", 30, 14, 1.0)], []), ("중금속 급증", [(4, "metal", 30, 14, 1.0)], []),
        ("소독 실패", [(0, "disinfect_fail", 30, 14, 1.0)], []), ("화학 오염(pH 급변)", [(2, "chemical", 30, 14, 1.0)], []),
        ("중금속 서서히 증가", [(5, "creeping_metal", 20, 24, 1.0)], []),
        ("하수+중금속 동시", [(7, "sewage", 30, 14, 1.0), (7, "metal", 34, 10, 1.0)], []),
        ("약한 하수 유입(강도 60%)", [(3, "sewage", 30, 14, 0.6)], []),
        ("교란: 탁도 센서 드리프트", [], [(1, "drift", "turbidity", 24, 18)]),
        ("교란: 잔류염소 단발 스파이크", [], [(0, "spike", "chlorine", 40, 1)]),
        ("교란: 통신 단절(전 항목 결측)", [], [(6, "dropout", "ph", 30, 10)])]
rows = []
for name, ev, ft in SCEN:
    c = prepare(make_dataset(days=3, seed=21, events=ev, faults=ft))
    c["ml"] = model.predict(c[FEATS])
    if ev:
        sid = SOURCES[ev[0][0]][0]; on = ev[0][2]
        hit = c[(c["source_id"] == sid) & (c["ml"] == 2) & (c["t"] >= on)]
        lag = int(hit["t"].min() - on) if len(hit) else None
        ok = lag is not None and lag <= 8
        rows.append((name, "오염", f"{lag}h" if lag is not None else "미탐", "PASS" if ok else "FAIL"))
    else:
        sid = SOURCES[ft[0][0]][0]
        n = int(((c["source_id"] == sid) & (c["ml"] == 2)).sum())
        rows.append((name, "교란", f"위험 오경보 {n}h", "PASS" if n <= 2 else "FAIL"))
res = pd.DataFrame(rows, columns=["시나리오", "구분", "탐지 지연/오경보", "판정"])
print("\n[복합 위협 시나리오 10종]")
print(res.to_string(index=False))
n_pass = int((res["판정"] == "PASS").sum())
print(f"통과 {n_pass}/10 (오염: 시작 후 8시간 내 위험 판정, 교란: 위험 오경보 2시간 이하)")

plt.figure(figsize=(8, 4))
imp.iloc[::-1].plot.barh(color="tab:blue")
plt.title("Random forest feature importance (top 6, synthetic)"); plt.xlabel("importance")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 04 예상 실행 결과 (Output)</b></summary>

```text
[교육용가상자료] 학습 5,376건 / 평가 2,688건, 특징 29개
Baseline 기준치: 정확도 97.9%, 위험 재현율 88.6%, 위험 오경보 17건
랜덤 포레스트: 정확도 99.3%, 위험 재현율 97.2%, 위험 오경보 10건
복합 위협 재현율 97.2% (목표 85% 이상: 달성), 처리 지연 1건 20.7ms / 전체 49ms (목표 200ms 이하: 달성)
Baseline이 놓친 서서히 오르는 중금속(creeping_metal) 중 모델 검출: 17/22시간
모델 위험 오경보의 센서 고장 유형: {'drift': 10} -> drift(서서히 틀어짐)는 미션 5 교차 검증으로 보완
중요 특징 Top6: storm 0.17, turbidity 0.15, metal 0.10, turbidity_dev 0.09, ph 0.07, river 0.06

[복합 위협 시나리오 10종]
              시나리오 구분  탐지 지연/오경보   판정
             하수 유입 오염         0h PASS
            중금속 급증 오염         0h PASS
             소독 실패 오염         3h PASS
      화학 오염(pH 급변) 오염         0h PASS
        중금속 서서히 증가 오염         2h PASS
         하수+중금속 동시 오염         0h PASS
  약한 하수 유입(강도 60%) 오염         0h PASS
    교란: 탁도 센서 드리프트 교란 위험 오경보 13h FAIL
  교란: 잔류염소 단발 스파이크 교란  위험 오경보 0h PASS
교란: 통신 단절(전 항목 결측) 교란  위험 오경보 0h PASS
통과 9/10 (오염: 시작 후 8시간 내 위험 판정, 교란: 위험 오경보 2시간 이하)
```
</details>
